# 01 — Exploração e análise descritiva (EDA)

Estatísticas descritivas, visualizações e matriz de correlação.

## Introdução

### Contexto

O Brasil registra mais de **2,5 milhões de nascimentos por ano**. Cada nascimento gera uma Declaração de Nascido Vivo (DN), compilada pelo DATASUS no **SINASC** — Sistema de Informações sobre Nascidos Vivos. Esse dataset reúne informações sobre a mãe, a gestação, o parto e o recém-nascido.

### Problema

Nascimentos prematuros (antes de 37 semanas de gestação) são a principal causa de mortalidade neonatal no mundo. No Brasil, representam cerca de **11% dos nascimentos** e concentram a maioria das internações em UTI neonatal. A prematuridade está associada a condições maternas identificáveis ainda durante o pré-natal — o que abre espaço para intervenção precoce.

### Objetivo

Construir um modelo preditivo capaz de **estimar o risco de prematuridade** a partir de informações disponíveis **antes ou durante o pré-natal** (perfil da mãe, histórico obstétrico, acompanhamento pré-natal e localização geográfica). O modelo deve ser acionável em Unidades Básicas de Saúde (UBS) para busca ativa de gestantes em risco.

### Por que prematuridade e não baixo peso?

Quase todo prematuro apresenta baixo peso, mas nem todo recém-nascido com baixo peso é prematuro — pode ser restrição de crescimento intrauterino (RCIU) a termo. São desfechos sobrepostos, mas não idênticos. Para busca ativa, **prematuridade é mais acionável**: identificado o risco precocemente, é possível adiar o parto com intervenções concretas (corticoide antenatal, repouso, controle de pressão, tocólise). Um modelo que prediz prematuridade funciona **indiretamente como proteção contra baixo peso**, atacando a raiz do problema.

### Dados utilizados

- **Fonte:** SINASC / DATASUS via `pysus`
- **Escopo:** Declarações de Nascidos Vivos (DN) — Minas Gerais, 2020–2022
- **Volume:** ~549 mil registros, 64 colunas originais
- **Target:** `PREMATURO = 1` se `SEMAGESTAC < 37`, 0 caso contrário

## 1. Imports

In [ ]:
# Imports (ajustar conforme o dataset)
import time
import pandas as pd
from pysus import SINASC
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

## 2. Conexão com o DATASUS / SINASC

Conecta à API do SINASC via `pysus` e inspeciona os grupos e metadados disponíveis.

In [ ]:
sinasc = SINASC().load() # Loads the files from DATASUS

In [ ]:
sinasc.metadata

In [ ]:
sinasc.groups

## 3. Download dos Dados

Baixa as Declarações de Nascidos Vivos (DN) do estado de **Minas Gerais** para os anos de **2020, 2021 e 2022** diretamente do DATASUS.
Os arquivos `.dbc` são convertidos automaticamente para `.parquet` e salvos em `../data/`.

In [ ]:
files = sinasc.get_files("DN", uf=["MG"], year=[2020, 2021, 2022])
files

In [ ]:
parquets = []
for f in files:
    parquet = sinasc.download([f], local_dir='../data')
    parquets.append(parquet)
    print(f"Baixado: {f}")
    time.sleep(5)

## 4. Carregamento e Merge

Lê os arquivos `.parquet` salvos, concatena os três anos em um único DataFrame e adiciona coordenadas geográficas (latitude/longitude) via merge com a tabela de municípios do IBGE.

In [ ]:
from pathlib import Path

parquets = list(Path("../data").glob("DNMG*.parquet"))

if not parquets:
    raise FileNotFoundError("Nenhum arquivo .parquet encontrado em ../data. Execute a célula de download primeiro.")

In [ ]:
df = pd.concat([pd.read_parquet(f) for f in parquets], ignore_index=True)

### Merge com coordenadas geográficas (IBGE)

O SINASC registra apenas o **código IBGE do município de residência** (`CODMUNRES`), um inteiro de 6 dígitos. Usar esse código diretamente como feature seria problemático: o modelo trataria o código como grandeza numérica ordinal (313375 > 310020), quando na verdade é apenas um identificador nominal sem ordem significativa.

A solução é converter o código em **latitude e longitude**, que:

- São variáveis contínuas com **significado espacial real** — municípios próximos têm coordenadas próximas
- Funcionam como **proxy socioeconômico**: regiões Sul/Sudeste de MG têm IDH, renda e acesso a serviços diferentes do Norte/Vale do Jequitinhonha, e o modelo pode capturar esses padrões implicitamente
- Permitem futuramente usar **features de vizinhança** (clustering geoespacial, distância a centros hospitalares de referência)

> **Detalhe técnico do join:** o código IBGE na tabela de municípios tem 7 dígitos (inclui dígito verificador), enquanto `CODMUNRES` tem 6. Por isso o merge é feito com `munic["codigo_ibge"] // 10` — dividindo por 10 para remover o último dígito e alinhar as chaves.

In [ ]:
df["CODMUNRES"] = pd.to_numeric(df["CODMUNRES"], errors="coerce")

munic = pd.read_csv(
    "../data/municipios_ibge.csv"
)

df = df.merge(
    munic[["codigo_ibge", "latitude", "longitude"]],
    left_on="CODMUNRES",
    right_on=munic["codigo_ibge"] // 10,
    how="left"
)

# # Padronização de nomes de colunas para maiúsculos
df.columns = df.columns.str.upper()

## 5. Análise Exploratória (EDA)

Visão geral do dataset: dimensões, tipos, valores nulos, estatísticas descritivas e conversão de colunas para tipos numéricos.

In [ ]:
df.shape

In [ ]:
df.describe()

### Conversão de tipos: por que todas as colunas chegam como `object`?

O SINASC distribui os dados em formato `.dbc` (dBase compactado), um formato legado do DATASUS. Ao converter para parquet via `pysus`, **todas as colunas são lidas como string (`object`)** — incluindo campos que semanticamente são numéricos, como idade, peso e semanas de gestação.

Isso acontece porque o `.dbc` não tem tipagem forte: o campo `IDADEMAE = "30"` é armazenado como texto, não como inteiro.

O bloco abaixo força a conversão das colunas que serão usadas como numéricas com `pd.to_numeric(..., errors="coerce")`:
- Valores que são de fato números (`"30"`, `"2815"`) são convertidos normalmente
- Valores inválidos ou sentinelas textuais (`""`, `" "`, `"99"` que virou texto corrompido) viram `NaN` automaticamente — o argumento `errors="coerce"` garante isso sem lançar exceção

> Colunas **não** listadas aqui (como `CODOCUPMAE`, `DTNASC`, flags administrativas) permanecem como string pois serão tratadas como categóricas ou descartadas.

In [ ]:
num_cols = [
    "IDADEMAE", "PESO", "APGAR1", "APGAR5", "QTDFILVIVO", "QTDFILMORT",
    "GESTACAO", "GRAVIDEZ", "PARTO", "CONSULTAS", "SEXO", "RACACOR",
    "IDANOMAL", "ESCMAE", "ESCMAE2010", "CONSPRENAT",
    "MESPRENAT", "QTDGESTANT", "QTDPARTNOR", "QTDPARTCES", "IDADEPAI",
    "ESTCIVMAE", "LOCNASC", "SEMAGESTAC"
]

# Deduplicar colunas
num_cols = list(dict.fromkeys(num_cols))

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [ ]:
df[num_cols].describe()

In [ ]:
df[num_cols].head()

## 6. Seleção de Features e Definição do Target

### Target: prematuridade (`SEMAGESTAC < 37`)

`SEMAGESTAC` e `GESTACAO` registram a idade gestacional no momento do parto sendo, o próprio desfecho que queremos prever. **Não podem aparecer como features em nenhuma etapa do pipeline** (leakage direto).

`PESO` também é excluído das features: baixo peso e prematuridade são altamente correlacionados (quase todo prematuro tem baixo peso), o que configuraria **leakage inverso**: o modelo aprenderia a prever o alvo a partir de uma consequência quase direta dele.

---

### Features selecionadas

**Perfil da mãe** (disponível desde a 1ª consulta):
`IDADEMAE`, `ESCMAE2010`, `ESTCIVMAE`, `RACACORMAE`, `QTDGESTANT`, `QTDPARTNOR`, `QTDPARTCES`, `QTDFILVIVO`, `QTDFILMORT`

**Gestação atual** (disponível durante pré-natal):
`GRAVIDEZ` (única/dupla/tripla), `CONSPRENAT` (nº consultas realizadas), `MESPRENAT` (mês de início do pré-natal), `SEXO` (via USG)

**Geográficas** (proxy socioeconômico — ver seção 4):
`LATITUDE`, `LONGITUDE`

**Perfil do pai:**
`IDADEPAI` — mantida apesar dos muitos nulos (~50%). O nulo aqui **não é missing aleatório**: pai ausente ou não declarado é um marcador de vulnerabilidade social associado a menor acesso ao pré-natal e maior risco de prematuridade. Estratégia: criar feature `PAI_AUSENTE` (1 se nulo, 0 caso contrário) e imputar a mediana nos preenchidos.

---

### O que foi descartado e por quê

| Motivo | Colunas |
|--------|---------|
| **Target** (desfecho a prever) | `SEMAGESTAC`, `GESTACAO` |
| **Leakage inverso** (consequência direta do target) | `PESO` — baixo peso e prematuridade são altamente correlacionados |
| **Leakage** (só disponível no parto) | `APGAR1/5`, `TPROBSON`, `TPAPRESENT`, `STTRABPART`, `STCESPARTO`, `KOTELCHUCK` |
| **Nulos excessivos sem sinal** (>30%, ausência não interpretável) | `DTRECORIGA`, `CODANOMAL`, `DTULTMENST`, `SERIESCMAE` |
| **Redundantes** | `ESCMAE` (→ `ESCMAE2010`), `CONSULTAS` (→ `CONSPRENAT`) |
| **Administrativas** | `ORIGEM`, `CODESTAB`, `NUMEROLOTE`, `VERSAOSIST`, `CONTADOR`, datas de sistema |

---

### Plano de tratamento

1. Strings vazias → `NaN`
2. Conversão de tipos (`pd.to_numeric`)
3. Sentinelas → `NaN` (9, 99, 999, 9999 = "ignorado" no DATASUS)
4. Filtros biológicos (idade mãe < 10 ou > 55)
5. Feature engineering: `PRIMIPARA`, `TEM_PERDA_FETAL`, `FAIXA_ETARIA`, `PN_TARDIO`, `PAI_AUSENTE`
6. Imputação: mediana (numéricas), `"nao_informado"` (categóricas)
7. One-hot encoding → split 80/20 estratificado → `StandardScaler`

In [ ]:
cols_prenatal = [
    # Target (criado abaixo a partir de SEMAGESTAC — não entra como feature)
    "SEMAGESTAC", "GESTACAO",
    # "PESO" — leakage inverso (consequência da prematuridade)

    # Perfil da mãe (disponível desde a 1ª consulta)
    "IDADEMAE", "ESCMAE2010", "ESTCIVMAE", "RACACORMAE",
    "QTDGESTANT", "QTDPARTNOR", "QTDPARTCES",
    "QTDFILVIVO", "QTDFILMORT",
    # Gestação atual (disponível durante pré-natal)
    "GRAVIDEZ",        # única/dupla/tripla
    "MESPRENAT",       # quando começou pré-natal
    "CONSPRENAT",      # nº consultas realizadas
    "SEXO",            # USG revela
    # Info pai
    "IDADEPAI",
    # Geo (proxy socioeconômico)
    "LATITUDE", "LONGITUDE",
]

df_model = df[cols_prenatal].copy()
semagestac = pd.to_numeric(df["SEMAGESTAC"], errors="coerce")
mask_valida = semagestac.notna()

df_model = df.loc[mask_valida, cols_prenatal].copy()
df_model["PREMATURO"] = (semagestac.loc[mask_valida] < 37).astype("int8")


### Pai ausente como marcador de vulnerabilidade social

O campo `IDADEPAI` tem **~54,8% de valores nulos** (300.888 de 549.334 registros) — e esse nulo não é aleatório. No SINASC, a idade do pai só é preenchida se ele estiver presente na declaração de nascido vivo, o que exige presença física ou reconhecimento formal da paternidade. A ausência reflete, portanto, uma combinação de fatores sociais: mães sem parceiro estável, relações não formalizadas, ou simplesmente pai não declarado no momento do registro.

Essa distinção importa para o modelo: tratar o nulo como dado faltante simples seria perda de informação. O nulo **carrega sinal** — gestações sem figura paterna tendem a estar associadas a menor suporte social, menor acesso ao pré-natal e maior risco de prematuridade. Por isso, a estratégia adotada é dupla:

1. **Criar a feature binária `PAI_AUSENTE`** (1 = IDADEPAI nulo, 0 = preenchido), preservando o sinal social antes de qualquer imputação
2. **Imputar a mediana global em todos os nulos de `IDADEPAI`**, permitindo que a coluna seja usada como feature contínua sem introduzir viés de escala

O resultado da distribuição nos histogramas abaixo confirma que pai ausente é a **condição majoritária** no dataset: mais da metade dos nascimentos em MG entre 2020 e 2022 não têm a idade do pai registrada.

In [ ]:
# Filtro de sanidade para IDADEPAI
# Removemos valores absurdos que distorcem a mediana
df_model.loc[df_model['IDADEPAI'] > 80, 'IDADEPAI'] = np.nan

df_model["PAI_AUSENTE"] = df_model["IDADEPAI"].isna().astype(int)
df_model["IDADEPAI"] = df_model["IDADEPAI"].fillna(df_model["IDADEPAI"].median())

ESCMAE2010
ESTCIVMAE
RACACORMAE

## 7. Visualizações

Histogramas das features selecionadas, distribuição geográfica dos nascimentos em MG e matriz de correlação.

In [ ]:
df_model.shape

In [ ]:
df_model.head()

In [ ]:
info = pd.DataFrame({
    "dtype": df_model.dtypes,
    "non_null": df_model.notna().sum(),
    "null_pct": (df_model.isna().mean() * 100).round(1),
    "nunique": df_model.nunique(),
    "sample": df_model.iloc[0]
})
print(info.to_string())

### 7.2 Distribuição Geográfica

Começamos plotando os pontos de latitude e longitude dos registros sem nenhum fundo de mapa, apenas para entender como os dados se distribuem no espaço.

In [ ]:
df_model.plot(
    kind="scatter",
    x="LONGITUDE",
    y="LATITUDE",
    figsize=(5, 4),
    alpha=0.5,
    s=2
)

O conjunto de pontos forma um contorno reconhecível: o formato resultante corresponde ao estado de Minas Gerais. Isso nos dá confiança de que o merge com as coordenadas do IBGE foi feito corretamente e que não há registros com coordenadas fora do estado.

A partir disso, passamos para mapas interativos que nos permitem explorar padrões regionais com mais profundidade.

#### Mapas Interativos

Os mapas a seguir agregam os registros por célula geográfica. O tamanho de cada ponto representa o volume de partos naquela célula. Para garantir visibilidade mesmo em áreas com poucos registros, aplicamos um offset mínimo de `max(n) * 0.02` ao tamanho, preservando o valor real de `n` no hover. 

#### Mapa 1 — Taxa de Cesáreas por Região

In [ ]:
center = {'lat': df['LATITUDE'].mean(), 'lon': df['LONGITUDE'].mean()}

df['cesarea'] = (df['PARTO'] == 2).astype(int)
df_agrupado = agrupado = df.assign(
    lat_round=df['LATITUDE'],
    lon_round=df['LONGITUDE']
).groupby(['lat_round', 'lon_round']).agg(
    taxa_cesarea=('cesarea', 'mean'),
    n=('cesarea', 'count')
).reset_index()

# Tamanho mínimo do ponto
min_size = agrupado['n'].max() * 0.02  # 2% do maior ponto
agrupado['size_plot'] = agrupado['n'] + min_size 

fig = px.scatter_map(df_agrupado, 
                    lat='lat_round', lon='lon_round',
                    labels= {
                        'lat_round':'Latitude', 
                        'lon_round':'Longitude',
                        'taxa_cesarea': 'Percentual de Cesáreas (%)',
                        'n': 'Número de Cesáreas'
                    },
                    title="Distribuição de Percentuais de cesáreas em Minas Gerais (2020-2022)",
                    hover_data={'taxa_cesarea':":.1%", 'n': True, 'size_plot': False},
                    color='taxa_cesarea', size=agrupado['size_plot'],
                    color_continuous_scale='RdPu',
                    map_style='carto-darkmatter', 
                    zoom=4.5, center=center, 
                    )
fig.update_layout(
    coloraxis_colorbar=dict(
        tickformat=".0%"
    )
)

fig.show(renderer='notebook')

#### Mapa 2 — Risco de Prematuridade por Região

Aqui visualizamos a taxa média de prematuridade por célula geográfica (lat/lon arredondados a 1 casa decimal). A escala de cores diverge em torno da média estadual de **10,8%**: regiões em vermelho têm risco acima da média, regiões em verde ficam abaixo. O tamanho de cada ponto reflete o volume de partos registrados naquela área.

Uma pergunta natural ao olhar esse mapa é: as diferenças regionais são reais, ou simplesmente refletem onde há mais gente? Para responder isso, calculamos a taxa por mesorregião IBGE logo abaixo, separando o efeito de volume do efeito de risco.

In [ ]:
agrupado = df_model.assign(
    lat_round=df_model['LATITUDE'],
    lon_round=df_model['LONGITUDE']
).groupby(['lat_round', 'lon_round']).agg(
    risco_prematuro=('PREMATURO', 'mean'),
    n=('PREMATURO', 'count')
).reset_index()

# Tamanho mínimo do ponto
min_size = agrupado['n'].max() * 0.03  # 3% do maior ponto
agrupado['size_plot'] = agrupado['n'] + min_size


fig = px.scatter_map(agrupado, lat='lat_round', lon='lon_round',
                     color='risco_prematuro', 
                     size=agrupado['size_plot'],
                     color_continuous_scale='rdylgn_r',
                     color_continuous_midpoint=agrupado['risco_prematuro'].mean(),
                     range_color=[0, agrupado['risco_prematuro'].max()],
                     map_style='carto-darkmatter',
                     zoom=4.5, center=center,
                     labels= {
                            'lat_round':'Latitude', 
                            'lon_round':'Longitude',
                            'risco_prematuro': 'Percentual de Partos Prematuros (%)',
                            'n': 'Número de Partos Prematuros'
                        },
                     title= 'Distribuição de Partos Prematuros em Minas Gerais (2020-2022)',
                     hover_data={'risco_prematuro':":.1%", 'n': True, 'size_plot': False},
                    )

fig.update_layout(
    coloraxis_colorbar=dict(
        tickformat=".0%"
    )
)

fig.show(renderer='notebook')

In [ ]:
df_model.corr(numeric_only=True)

In [ ]:
%load_ext autoreload
%autoreload 2
from utils import save_if_different

file_path = "../data/df_model_raw.parquet"

save_if_different(df_model, file_path)